In [1]:
import plotly.io as pio

pio.renderers.default = "notebook"  # embeds interactive figures inline  [oai_citation:0‡plotly.com](https://plotly.com/python/renderers/?utm_source=chatgpt.com)

import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.express as px
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import squidpy as sq
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
from anndata import AnnData
import anndata as ad


def plot_adata(adata_to_plot, cluster_key, plot_type="umap", keyword=None):
    """
    Plot AnnData embeddings (UMAP or spatial) in Jupyter Notebook.
    """
    # Prepare DataFrame
    if plot_type == "spatial":
        coords = adata_to_plot.obsm['spatial']
        df = pd.DataFrame(coords, columns=['x','y'], index=adata_to_plot.obs_names)
    else:
        df = pd.DataFrame(
            adata_to_plot.obsm['X_umap'],
            columns=['UMAP1','UMAP2'],
            index=adata_to_plot.obs_names
        )

    df[cluster_key] = adata_to_plot.obs[cluster_key].astype(str)

    if keyword:
        df['plot_label'] = df[cluster_key].apply(lambda x: x if keyword in x else 'Other')
        color_key = 'plot_label'
        legend_title = f"Highlighted: {keyword}"
    else:
        color_key, legend_title = cluster_key, cluster_key

    # Discrete tab20 colors
    tab20 = [mpl.colors.rgb2hex(c) for c in plt.get_cmap('tab20').colors]

    # Build scatter
    if plot_type == "spatial":
        fig = px.scatter(
            df, x='x', y='y', color=color_key,
            hover_name=df.index, title='Spatial', 
            color_discrete_sequence=tab20,
            width=800, height=600
        )
        fig.update_yaxes(autorange='reversed')
    else:
        fig = px.scatter(
            df, x='UMAP1', y='UMAP2', color=color_key,
            hover_name=df.index, title='UMAP',
            color_discrete_sequence=tab20,
            width=800, height=600
        )

    fig.update_traces(marker=dict(size=6, opacity=0.8))
    fig.update_layout(
        legend_title_text=legend_title,
        legend=dict(bgcolor='rgba(255,255,255,0.5)', x=1.02, y=1),
        margin=dict(l=20, r=200, t=50, b=20),
        plot_bgcolor='white', paper_bgcolor='white'
    )

    # Force the notebook renderer
    fig.show(renderer="notebook")  # ensures inline display  [oai_citation:1‡saturncloud.io](https://saturncloud.io/blog/troubleshooting-plotly-chart-not-showing-in-jupyter-notebook/?utm_source=chatgpt.com)


    import scanpy as sc
import gseapy as gp

def go_fgsea(adata, ref, comp, gene_set="MSigDB_Hallmark_2020", io_key = "assigned_celltype_L0", show_plot = True):
    Reference, Delta = adata[adata.obs[io_key] == ref], \
                       adata[adata.obs[io_key] == comp]
    
    adata_concat = sc.concat(
        [Reference, Delta],
        axis=0,
        join="inner",                # keep only shared variables
        label="subset",              # name of the new obs‐column
        keys=[ref, comp],
        merge="same"                 # assume var‐ and obsm‐entries are identical
    )

        # 1) Run differential expression: PartialTumor vs TumorEnriched
    sc.tl.rank_genes_groups(
    adata_concat,
    groupby='subset',
    groups=[comp],
    reference=ref,
    method='t-test'          # or 't-test', 'logreg', etc.
    )

    # 2) Extract a pre-ranked list of genes (log₂-fold-changes)
    de_df = sc.get.rank_genes_groups_df(
    adata_concat,
    group=comp,
    key='rank_genes_groups'
    )
    rnk = de_df.set_index('names')['logfoldchanges']

    pre_res = gp.prerank(
    rnk=rnk,
    gene_sets=gene_set,     # or path to your GMT file
    processes=4,
    permutation_num=1000,
    outdir=None)


    if show_plot:
        sc.pl.umap(
        adata_concat,
        color="subset",
        palette=["red","blue"],
        size=20,
        title=f"UMAP Comparisons of {ref} as reference vs {comp} as comparison",
        alpha = 0.4)

        terms = pre_res.res2d.Term
        axs = pre_res.plot(terms[:5], show_ranking=False, legend_kws={'loc': (1.05, 0)}, )

    return pre_res.res2d[pre_res.res2d["FDR q-val"] < 0.2] 


def de_volcano(
    adata,
    groupby: str,
    ref: str,
    comp: str,
    method: str = "t-test",
    pval_adj_method: str = "fdr_bh",
    lfc_thresh: float = 1.0,
    pval_thresh: float = 0.05,
    top_n: int = 10,
    show: bool = True
) -> pd.DataFrame:
    """
    Run group-level DE, display a volcano plot with gene labels for the top N hits.

    Returns a DataFrame with DE results including names, logfoldchanges, pvals, pvals_adj.
    """
    # 1) Subset and run DE
    ad = sc.concat(
        [adata[adata.obs[groupby] == ref],
         adata[adata.obs[groupby] == comp]],
        join="inner", label="subset", keys=[ref, comp], merge="same"
    )
    sc.tl.rank_genes_groups(
        ad, groupby="subset", groups=[comp], reference=ref, method=method
    )

    # 2) Build DE results table
    r = sc.get.rank_genes_groups_df(ad, group=comp)
    de_df = r[['names','logfoldchanges','pvals']].copy()
    de_df['pvals_adj'] = multipletests(de_df['pvals'], method=pval_adj_method)[1]

    if show:
        x = de_df['logfoldchanges']
        y = -np.log10(de_df['pvals_adj'] + 1e-300)

        plt.figure(figsize=(6,6))
        # all points
        plt.scatter(x, y, c='lightgrey', s=10, alpha=0.6)
        # significant
        sig = (np.abs(x) >= lfc_thresh) & (de_df['pvals_adj'] <= pval_thresh)
        plt.scatter(
            x[sig], y[sig],
            c=np.where(x[sig]>0, 'red','blue'),
            s=20, alpha=0.8
        )
        # threshold lines
        plt.axvline(lfc_thresh,  color='grey', linestyle='--', linewidth=1)
        plt.axvline(-lfc_thresh, color='grey', linestyle='--', linewidth=1)
        plt.axhline(-np.log10(pval_thresh), color='grey', linestyle='--', linewidth=1)

        # annotate top N by adjusted p-value
        top_hits = de_df.nsmallest(top_n, 'pvals_adj')
        for _, row in top_hits.iterrows():
            xi = row['logfoldchanges']
            yi = -np.log10(row['pvals_adj'] + 1e-300)
            plt.text(xi, yi, row['names'], fontsize=8,
                     ha='right' if xi<0 else 'left')

        plt.xlabel('log₂ fold-change')
        plt.ylabel('-log₁₀ adjusted p-value')
        plt.tight_layout()
        plt.show()

    return de_df

# Ensure inline rendering in Jupyter
pio.renderers.default = "notebook"

def plot_adata(adata_to_plot, cluster_key, plot_type="umap", keyword=None):
    """
    Plot AnnData embeddings (UMAP or spatial) in Jupyter Notebook, with
    the X axis extended to 3× its data span.
    """
    # Prepare DataFrame
    if plot_type == "spatial":
        coords = adata_to_plot.obsm['spatial']
        df = pd.DataFrame(coords, columns=['x','y'], index=adata_to_plot.obs_names)
        xcol, ycol = 'x', 'y'
        title = 'Spatial'
    else:
        df = pd.DataFrame(
            adata_to_plot.obsm['X_umap'],
            columns=['UMAP1','UMAP2'],
            index=adata_to_plot.obs_names
        )
        xcol, ycol = 'UMAP1', 'UMAP2'
        title = 'UMAP'

    df[cluster_key] = adata_to_plot.obs[cluster_key].astype(str)

    if keyword:
        df['plot_label'] = df[cluster_key].apply(lambda x: x if keyword in x else 'Other')
        color_key = 'plot_label'
        legend_title = f"Highlighted: {keyword}"
    else:
        color_key, legend_title = cluster_key, cluster_key

    # Discrete tab20 colors
    tab20 = [mpl.colors.rgb2hex(c) for c in plt.get_cmap('tab20').colors]

    # Build scatter
    fig = px.scatter(
        df,
        x=xcol,
        y=ycol,
        color=color_key,
        hover_name=df.index,
        title=title,
        color_discrete_sequence=tab20,
        width=1400,
        height=600
    )
    if plot_type == "spatial":
        fig.update_yaxes(autorange='reversed')

    fig.update_traces(marker=dict(size=6, opacity=0.8))
    fig.update_layout(
        legend_title_text=legend_title,
        legend=dict(bgcolor='rgba(255,255,255,0.5)', x=1.02, y=1),
        margin=dict(l=20, r=200, t=50, b=20),
        plot_bgcolor='white', paper_bgcolor='white'
    )

    # === New: extend X-axis to 3× its data span ===
    x_min, x_max = df[xcol].min(), df[xcol].max()
    span  = x_max - x_min
    center= (x_min + x_max) / 2
    half_new = 1.5 * span
    fig.update_xaxes(range=[center - half_new, center + half_new])

    # Force the notebook renderer
    fig.show(renderer="notebook")


def filter_adata_by_region(
    adata: AnnData,
    x_bounds: tuple[float, float],
    y_bounds: tuple[float, float],
    coord_key: str = "spatial"
) -> AnnData:
    """
    Subset AnnData to cells whose 2D coordinates lie within a rectangle.

    Parameters
    ----------
    adata
        Input AnnData with embedding or spatial coords in .obsm
    x_bounds
        (x_min, x_max) inclusive bounds on the first coordinate
    y_bounds
        (y_min, y_max) inclusive bounds on the second coordinate
    coord_key
        Key in adata.obsm containing an (n_obs, 2) array of coordinates
        e.g. "spatial" or "X_umap"

    Returns
    -------
    AnnData
        A new AnnData with only the cells inside the given region.
    """
    # Extract coords
    coords = adata.obsm.get(coord_key)
    if coords is None:
        raise KeyError(f"'{coord_key}' not found in adata.obsm")
    if coords.shape[1] < 2:
        raise ValueError(f"adata.obsm['{coord_key}'] must have at least 2 columns")

    x_min, x_max = x_bounds
    y_min, y_max = y_bounds

    # Build mask
    xs = coords[:, 0]
    ys = coords[:, 1]
    mask = (xs >= x_min) & (xs <= x_max) & (ys >= y_min) & (ys <= y_max)

    # Subset and return
    return adata[mask].copy()

def run_ligrec(adata):
    # all your parameters here
    return sq.gr.ligrec(
        adata,
        n_perms=200,
        cluster_key="assigned_celltype_L0",
        copy=True,
        use_raw=False,
        transmitter_params={"categories": "ligand"},
        receiver_params={"categories": "receptor"}
    )

def plot_ligrec(res,source,target):

    df_plot = res["means"][source][target].rename("value").reset_index()

    fig = px.scatter(
        df_plot,
        x="target",
        y="source",
        size="value",
        color="value",
        color_continuous_scale="Blues",
        size_max=20,
        hover_data=["value"],
        labels={"value": "Score"},
        width=1600,
        height=1000
    )
    fig.update_layout(
        xaxis_tickangle=45,
        yaxis=dict(tickfont=dict(size=8)),
        plot_bgcolor="white",
        margin=dict(l=200, r=20, t=50, b=200)
    )
    fig.show()





/Users/ugursahin/miniforge3/envs/ScanPy2/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning:

The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.

/Users/ugursahin/miniforge3/envs/ScanPy2/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.

/Users/ugursahin/miniforge3/envs/ScanPy2/lib/python3.12/site-packages/anndata/utils.py:434: FutureWarning:

Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.



In [2]:
import scanpy as sc
import squidpy as sq

Tissue = "Region1"

adata = sc.read_h5ad(f"/Volumes/ProstateCancerEvoMain/dbs/Completed/AllRegions/{Tissue}.raw.annotated.V2.h5ad")
adata


Tissue = "Region1"
TableFolder = "REGION1_TABLES"

adata_raw = sc.read_h5ad(f"/Volumes/ProstateCancerEvoMain/dbs/Ongoing/{Tissue}/{TableFolder}/{Tissue}_Xenium_Phen_HE_Integrated.GeneTranscripts_XStock_Native.V1.h5ad")
adata_raw

AnnData object with n_obs × n_vars = 113273 × 5101
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [3]:
adata.X = adata_raw.X

In [4]:
import numpy as np
from sklearn.cluster import KMeans
import math

def make_pseudo_tiles(adata, tile_size=75, coord_key="spatial", out_key="pseudoTILE"):
    """
    Partition cells into spatial clusters of ~tile_size cells each,
    and store labels in adata.obs[out_key].
    """
    # 1) extract XY coords
    coords = adata.obsm[coord_key]              # shape (n_cells, 2)
    
    # 2) decide how many clusters
    n_cells    = coords.shape[0]
    n_clusters = math.ceil(n_cells / tile_size)
    
    # 3) run KMeans
    km = KMeans(n_clusters=n_clusters, random_state=0)
    labels = km.fit_predict(coords)             # integers 0..n_clusters-1
    
    # 4) turn into human‐friendly names
    pseudo_names = [f"pseudoTILE_{i+1}" for i in labels]
    
    # 5) assign back to adata.obs
    adata.obs[out_key] = pseudo_names
    adata.obs[out_key] = adata.obs[out_key].astype("category")
    
    return adata

# usage


In [5]:
import pandas as pd
import scipy.sparse as sp

def collect_into_pseudobulk(adata, target_col="pseudoTILE", pseudo_group="pseudoTILE"):
    # 1) Build and assign the grouping key
    grp = (
        adata.obs["assigned_celltype_L0"].astype(str)
        + "_"
        + adata.obs[target_col].astype(str)
    )
    adata.obs[pseudo_group] = grp.astype("category")

    # 2) Extract counts into a DataFrame
    X = adata.X
    if sp.issparse(X):
        df = pd.DataFrame.sparse.from_spmatrix(
            X,
            index=adata.obs_names,
            columns=adata.var_names
        )
    else:
        df = pd.DataFrame(
            X,
            index=adata.obs_names,
            columns=adata.var_names
        )

    # 3) Attach pseudo_group and sum
    df[pseudo_group] = adata.obs[pseudo_group]
    agg = df.groupby(pseudo_group).sum()      # rows = pseudo_groups, cols = genes

    # 4) Transpose so rows = genes, cols = pseudo_groups
    agg_counts = agg.T
    agg_counts.index.name = "gene"

    return agg_counts

In [6]:
tumor_adata = adata[adata.obs["assigned_celltype_L0"] == "tumor_cell_markers"]

make_pseudo_tiles(tumor_adata)
tumor_adata_pseudobulk = collect_into_pseudobulk(tumor_adata)

/var/folders/4p/7h_929nx5qngrf9z7xhfy15w0000gn/T/ipykernel_84076/324497109.py:25: ImplicitModificationWarning:

Trying to modify attribute `.obs` of view, initializing view as actual.

/var/folders/4p/7h_929nx5qngrf9z7xhfy15w0000gn/T/ipykernel_84076/3476175776.py:30: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [7]:
basal_epithelial_cell_of_prostatic_duct = adata[adata.obs["assigned_celltype_L0"] == "basal_epithelial_cell_of_prostatic_duct"]

make_pseudo_tiles(basal_epithelial_cell_of_prostatic_duct)
basal_epithelial_cell_of_prostatic_duct_pseudobulk  = collect_into_pseudobulk(basal_epithelial_cell_of_prostatic_duct)

/var/folders/4p/7h_929nx5qngrf9z7xhfy15w0000gn/T/ipykernel_84076/324497109.py:25: ImplicitModificationWarning:

Trying to modify attribute `.obs` of view, initializing view as actual.

/var/folders/4p/7h_929nx5qngrf9z7xhfy15w0000gn/T/ipykernel_84076/3476175776.py:30: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [8]:
basal_cell_of_prostate_epithelium = adata[adata.obs["assigned_celltype_L0"] == "basal_cell_of_prostate_epithelium"]

make_pseudo_tiles(basal_cell_of_prostate_epithelium)
basal_cell_of_prostate_epithelium_pseudobulk = collect_into_pseudobulk(basal_cell_of_prostate_epithelium)

/var/folders/4p/7h_929nx5qngrf9z7xhfy15w0000gn/T/ipykernel_84076/324497109.py:25: ImplicitModificationWarning:

Trying to modify attribute `.obs` of view, initializing view as actual.

/var/folders/4p/7h_929nx5qngrf9z7xhfy15w0000gn/T/ipykernel_84076/3476175776.py:30: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [9]:
basal_cell_of_prostate_epithelium_pseudobulk.index

Index(['A2ML1', 'AAMP', 'AAR2', 'AARSD1', 'ABAT', 'ABCA1', 'ABCA3', 'ABCA4',
       'ABCA7', 'ABCB1',
       ...
       'ZPR1', 'ZSCAN1', 'ZSCAN12', 'ZSCAN16', 'ZSCAN20', 'ZSCAN26', 'ZSWIM6',
       'ZUP1', 'ZYG11B', 'ZYX'],
      dtype='object', name='gene', length=5101)

In [10]:
# List your pseudobulk DataFrames
dfs = [
    basal_cell_of_prostate_epithelium_pseudobulk,
    basal_epithelial_cell_of_prostatic_duct_pseudobulk,
    tumor_adata_pseudobulk
]

merged_pseudobulk = pd.concat(dfs, axis=1, join="inner")
merged_pseudobulk

pseudoTILE,basal_cell_of_prostate_epithelium_pseudoTILE_1,basal_cell_of_prostate_epithelium_pseudoTILE_10,basal_cell_of_prostate_epithelium_pseudoTILE_100,basal_cell_of_prostate_epithelium_pseudoTILE_101,basal_cell_of_prostate_epithelium_pseudoTILE_102,basal_cell_of_prostate_epithelium_pseudoTILE_103,basal_cell_of_prostate_epithelium_pseudoTILE_104,basal_cell_of_prostate_epithelium_pseudoTILE_105,basal_cell_of_prostate_epithelium_pseudoTILE_106,basal_cell_of_prostate_epithelium_pseudoTILE_107,...,tumor_cell_markers_pseudoTILE_90,tumor_cell_markers_pseudoTILE_91,tumor_cell_markers_pseudoTILE_92,tumor_cell_markers_pseudoTILE_93,tumor_cell_markers_pseudoTILE_94,tumor_cell_markers_pseudoTILE_95,tumor_cell_markers_pseudoTILE_96,tumor_cell_markers_pseudoTILE_97,tumor_cell_markers_pseudoTILE_98,tumor_cell_markers_pseudoTILE_99
gene,,,,,,,,,,,,,,,,,,,,,
A2ML1,0,1.0,0,0,0,0,0,0,0,0,...,1.0,0,0,0,0,0,0,1.0,0,0
AAMP,11.0,2.0,0,14.0,9.0,8.0,9.0,0,10.0,7.0,...,14.0,5.0,0,2.0,26.0,7.0,22.0,5.0,5.0,12.0
AAR2,3.0,0,0,1.0,0,1.0,3.0,0,2.0,0,...,3.0,1.0,0,0,3.0,2.0,7.0,3.0,2.0,0
AARSD1,1.0,1.0,0,2.0,0,3.0,1.0,0,3.0,3.0,...,5.0,3.0,0,0,4.0,1.0,7.0,2.0,1.0,7.0
ABAT,0,0,0,0,0,0,0,0,0,0,...,6.0,5.0,0,0,4.0,4.0,12.0,4.0,4.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSCAN26,0,4.0,0,1.0,1.0,2.0,1.0,0,2.0,1.0,...,2.0,0,0,0,2.0,3.0,1.0,0,1.0,0
ZSWIM6,1.0,1.0,0,0,0,2.0,0,0,1.0,1.0,...,4.0,1.0,0,0,2.0,0,7.0,1.0,0,2.0
ZUP1,1.0,0,0,1.0,1.0,4.0,1.0,0,2.0,0,...,2.0,2.0,0,0,3.0,0,7.0,1.0,1.0,0


In [11]:
ann =pd.DataFrame([[el,el.split("_pseudo")[0]] for el in merged_pseudobulk.columns.to_list()])

In [12]:
ann.to_csv("R1.ann.tsv", sep="\t")

In [13]:
merged_pseudobulk.to_csv("R1.pseudobulk.healthy_tumor.tsv", sep="\t")